In [1]:

import heapq

def uniform_cost_search(initial_state, goal_test, get_successors):
    """
    Generic Uniform-Cost Search.
    Returns: (goal_state, action_sequence, state_sequence, total_cost) or None
    """
    counter = 0  # tie-breaker for the heap (states may not be directly comparable)
    frontier = [(0, counter, initial_state, [], [initial_state])]
    visited = set()

    while frontier:
        cost, _, state, actions, path = heapq.heappop(frontier)

        if state in visited:
            continue
        visited.add(state)

        if goal_test(state):
            return state, actions, path, cost

        for action, next_state, step_cost in get_successors(state):
            if next_state not in visited:
                counter += 1
                heapq.heappush(
                    frontier,
                    (cost + step_cost, counter, next_state, actions + [action], path + [next_state])
                )

    return None  # no solution found


In [2]:

# --- 1. Warehouse layout: grid coordinates for every location ---
positions = {
    "Entrance":     (0, 0),
    "Aisle-1":      (0, 1),
    "Aisle-2":      (0, 2),
    "Rack-12":      (0, 3),
    "Zone-B-Entry": (1, 1),
    "Rack-7":       (1, 2),
    "Rack-9":       (2, 1),
    "Rack-20":      (2, 2),
}

# --- 2. Warehouse aisles: undirected weighted edges (distance in meters) ---
raw_edges = [
    ("Entrance",     "Aisle-1",      4),
    ("Aisle-1",      "Aisle-2",      3),
    ("Aisle-2",      "Rack-12",      2),
    ("Aisle-1",      "Zone-B-Entry", 5),
    ("Zone-B-Entry", "Rack-7",       3),
    ("Zone-B-Entry", "Rack-9",       6),
    ("Rack-7",       "Rack-20",      3),
    ("Rack-9",       "Rack-20",      2),   # creates a cycle -> tests visited-state handling
]

def direction_of(a, b):
    """Cardinal direction a robot moves in to go from node a to node b."""
    (x1, y1), (x2, y2) = positions[a], positions[b]
    dx, dy = x2 - x1, y2 - y1
    if dx == 1:  return "Move East"
    if dx == -1: return "Move West"
    if dy == 1:  return "Move North"
    if dy == -1: return "Move South"
    raise ValueError(f"{a} and {b} are not axis-aligned neighbors")

# --- 3. Build the adjacency-list graph programmatically ---
warehouse_graph = {loc: [] for loc in positions}
for a, b, dist in raw_edges:
    warehouse_graph[a].append((direction_of(a, b), b, dist))
    warehouse_graph[b].append((direction_of(b, a), a, dist))

print("State-space graph (adjacency list): action -> (neighbor, distance)\n")
for loc, edges in warehouse_graph.items():
    print(f"{loc}:")
    for action, neighbor, dist in edges:
        print(f"    {action:<12} -> {neighbor} (distance={dist})")


State-space graph (adjacency list): action -> (neighbor, distance)

Entrance:
    Move North   -> Aisle-1 (distance=4)
Aisle-1:
    Move South   -> Entrance (distance=4)
    Move North   -> Aisle-2 (distance=3)
    Move East    -> Zone-B-Entry (distance=5)
Aisle-2:
    Move South   -> Aisle-1 (distance=3)
    Move North   -> Rack-12 (distance=2)
Rack-12:
    Move South   -> Aisle-2 (distance=2)
Zone-B-Entry:
    Move West    -> Aisle-1 (distance=5)
    Move North   -> Rack-7 (distance=3)
    Move East    -> Rack-9 (distance=6)
Rack-7:
    Move South   -> Zone-B-Entry (distance=3)
    Move East    -> Rack-20 (distance=3)
Rack-9:
    Move West    -> Zone-B-Entry (distance=6)
    Move North   -> Rack-20 (distance=2)
Rack-20:
    Move West    -> Rack-7 (distance=3)
    Move South   -> Rack-9 (distance=2)


In [3]:

def warehouse_successors(state):
    """Transition model: returns (action, next_state, cost) for every valid move."""
    return warehouse_graph[state]

def make_warehouse_goal_test(target_rack):
    def goal_test(state):
        return state == target_rack
    return goal_test


In [4]:

initial_state = "Entrance"
target_rack   = "Rack-20"
goal_test     = make_warehouse_goal_test(target_rack)

result = uniform_cost_search(initial_state, goal_test, warehouse_successors)

if result:
    goal_state, actions, path, total_cost = result
    print("Initial State :", initial_state)
    print("Goal State    :", goal_state)
    print("Search Algorithm Used: Uniform-Cost Search (UCS)")
    print("\nMovement Sequence:")
    for step_num, (action, loc) in enumerate(zip(actions, path[1:]), start=1):
        print(f"  {step_num}. {action:<12} -> arrive at {loc}")
    print(f"\nTotal Travel Distance: {total_cost} meters")
else:
    print("No route found to the target rack.")


Initial State : Entrance
Goal State    : Rack-20
Search Algorithm Used: Uniform-Cost Search (UCS)

Movement Sequence:
  1. Move North   -> arrive at Aisle-1
  2. Move East    -> arrive at Zone-B-Entry
  3. Move North   -> arrive at Rack-7
  4. Move East    -> arrive at Rack-20

Total Travel Distance: 15 meters


In [5]:

from collections import namedtuple

DeploymentState = namedtuple(
    "DeploymentState",
    ["resource_type", "vm_created", "dependencies_installed", "model_deployed", "verified"]
)

initial_deployment_state = DeploymentState(
    resource_type=None, vm_created=False, dependencies_installed=False,
    model_deployed=False, verified=False
)

def deployment_goal_test(state):
    return state.model_deployed and state.verified

def deployment_successors(state):
    """Transition model: enumerate every action whose precondition is met."""
    successors = []

    # Allocate Standard Resources
    if state.resource_type is None:
        successors.append((
            "Allocate Standard Resources",
            state._replace(resource_type="Standard"),
            2
        ))
    # Allocate High-Performance Resources
    if state.resource_type is None:
        successors.append((
            "Allocate High-Performance Resources",
            state._replace(resource_type="HighPerformance"),
            4
        ))
    # Create Standard Virtual Machine
    if state.resource_type == "Standard" and not state.vm_created:
        successors.append((
            "Create Standard Virtual Machine",
            state._replace(vm_created=True),
            3
        ))
    # Create High-Performance Virtual Machine
    if state.resource_type == "HighPerformance" and not state.vm_created:
        successors.append((
            "Create High-Performance Virtual Machine",
            state._replace(vm_created=True),
            5
        ))
    # Install Dependencies
    if state.vm_created and not state.dependencies_installed:
        successors.append((
            "Install Dependencies",
            state._replace(dependencies_installed=True),
            2
        ))
    # Deploy AI Model
    if state.dependencies_installed and not state.model_deployed:
        successors.append((
            "Deploy AI Model",
            state._replace(model_deployed=True),
            3
        ))
    # Verify Deployment
    if state.model_deployed and not state.verified:
        successors.append((
            "Verify Deployment",
            state._replace(verified=True),
            1
        ))

    return successors


In [6]:

def generate_reachable_states(initial_state, get_successors):
    seen = {initial_state}
    frontier = [initial_state]
    edges = []
    while frontier:
        current = frontier.pop()
        for action, nxt, cost in get_successors(current):
            edges.append((current, action, nxt, cost))
            if nxt not in seen:
                seen.add(nxt)
                frontier.append(nxt)
    return seen, edges

reachable_states, reachable_edges = generate_reachable_states(
    initial_deployment_state, deployment_successors
)

print(f"Total reachable states: {len(reachable_states)}\n")
print("State-space graph (state -> action -> next_state [cost]):\n")
for src, action, dst, cost in reachable_edges:
    print(f"{src}\n   --{action} [cost={cost}]-->\n{dst}\n")


Total reachable states: 11

State-space graph (state -> action -> next_state [cost]):

DeploymentState(resource_type=None, vm_created=False, dependencies_installed=False, model_deployed=False, verified=False)
   --Allocate Standard Resources [cost=2]-->
DeploymentState(resource_type='Standard', vm_created=False, dependencies_installed=False, model_deployed=False, verified=False)

DeploymentState(resource_type=None, vm_created=False, dependencies_installed=False, model_deployed=False, verified=False)
   --Allocate High-Performance Resources [cost=4]-->
DeploymentState(resource_type='HighPerformance', vm_created=False, dependencies_installed=False, model_deployed=False, verified=False)

DeploymentState(resource_type='HighPerformance', vm_created=False, dependencies_installed=False, model_deployed=False, verified=False)
   --Create High-Performance Virtual Machine [cost=5]-->
DeploymentState(resource_type='HighPerformance', vm_created=True, dependencies_installed=False, model_deployed=Fal

In [7]:

result = uniform_cost_search(
    initial_deployment_state, deployment_goal_test, deployment_successors
)

if result:
    goal_state, actions, path, total_cost = result
    print("Initial State :", initial_deployment_state)
    print("Goal State    :", goal_state, " (Completed and Verified)")
    print("Search Algorithm Used: Uniform-Cost Search (UCS)")
    print("\nSelected Deployment Sequence:")
    for step_num, action in enumerate(actions, start=1):
        print(f"  {step_num}. {action}")
    print(f"\nTotal Path Cost: {total_cost}")
else:
    print("No valid deployment plan found.")


Initial State : DeploymentState(resource_type=None, vm_created=False, dependencies_installed=False, model_deployed=False, verified=False)
Goal State    : DeploymentState(resource_type='Standard', vm_created=True, dependencies_installed=True, model_deployed=True, verified=True)  (Completed and Verified)
Search Algorithm Used: Uniform-Cost Search (UCS)

Selected Deployment Sequence:
  1. Allocate Standard Resources
  2. Create Standard Virtual Machine
  3. Install Dependencies
  4. Deploy AI Model
  5. Verify Deployment

Total Path Cost: 11
